In [1]:
import random
import math
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

C:\Users\s1604058\AppData\Roaming\Python\Python39\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\s1604058\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In this notebook, we take the raw data produced by our participants at test and extract the final kinship systems they produced. From there, we can calculate the symmetric conditional entropy of their final languages, create simulated baselines against which we compare their true behaviour, and measure their 'accuracy': how often they made predictive generalisations.

First, let's import the dataframe containing all participants' responses.

In [2]:
all_data = pd.read_csv('data/all_data.csv')

And get a list of random IDs associated with each participant.

In [3]:
random_ids = list(set(list(all_data['random_id'])))

Next, we want to extract participants' data individually, so we can look at their final languages.


In [4]:
# get one dataframe

def get_ppt_df(ppt_id):
   
    df = all_data.drop(all_data[all_data.random_id != ppt_id].index)
    
    df.drop(index = df.index[0])
    
    # remove any rows where participants pressed the 'done' and 'undo' buttons
    df = df.drop(df[df.done == 1].index)
    df = df.drop(df[df.undo == 1].index)
    
    # filter any participant who failed more than 1 catch trial
    if len(df[df.catch_failed == True]) < 2:
        return df

And to extract the kinship system they produced into a dictionary format:

In [5]:
referents = ['mzd','mzs','mbd','mbs','z','b','fzd','fzs','fbd','fbs']

In [6]:
def get_ppt_ks(df):
        
    ks = {} # an empty dict to start
    
    choices = []
    
    for i in range(len(df)): # for each line in their dataframe
        relative = df['selectee'].values[i] # find out who the referent was for that trial
        
        if df['trial_type'].values[i] == 'catch': # ignore if it's a catch trial
            continue

        else:
            term = df['selection'].values[i] # find out what term they selected
            ks[relative] = term # add referent and term to dict
    
    # find out what the given G0 terms were for this participant and add referent and term to dict
    generaliser1 = df['generaliser1'].values[0] 
    ks[generaliser1] = df['generaliser1_term'].values[0]
    generaliser2 = df['generaliser2'].values[0]
    ks[generaliser2] = df['generaliser2_term'].values[0]
    
    return ks
    

Now we have a record of what labels our participants chose, let's build up the rest of the kinship system (i.e. what terms they were given in G+1, which are the same for every participant but not saved in the dataframe.)

In [7]:
# these lists represent the eight trial types in the experiment
# individuals in the same sublist share a term
language_types = [[['m'],['f'],['mz','mb','fz','fb']], # Type I
                [['m'],['f'],['mz','mb'],['fz','fb'],['z','b']], # Type IIside
                [['m'],['f'],['mz','fz'],['mb','fb'],['z','b']], # Type II gen
                [['m'],['f'],['mz','fb'],['mb','fz'],['z','b']], # Type II opp
                [['m'],['f'],['mz','mb','fz'],['fb'],['z','b']], # Type III fb
                [['m'],['f'],['mz','mb','fb'],['fz'],['z','b']], # Type III fz
                [['m'],['f'],['mz','fz','fb'],['mb'],['z','b']], # Type III mb
                [['m'],['f'],['fz','mb','fb'],['mz'],['z','b']]] # Type III mz

# given a participants' responses, add the rest of the kinship system to dict
def make_ks(language_type,ks):
    """ A language_type is a list of lists representing how terms are spread across the space of relatives. 
    Every relative in a sublist shares a term. This function takes that spread and adds to an existing dictionary where
    keys are relatives and values are terms."""
            
    for group in language_type: # for each group of relatives that share a term
        for relative in group: # for each relative in that group
            ks[relative] = str(language_type.index(group)) # assign them a value equal to the group's index (i.e. a unique 'term')

    return ks

## Proportion of predictive generalisations

For each participant, let's code how often they made a generalisation that increased the predictive structure of the system *at that point*. i.e., we want to assign them 1 if their choice led to the lowest symmetric conditional entropy of all four choices, or 0 if it led to a higher symmetric conditional entropy.

First, we need infrastructure for calculate symmetric conditional entropy.

In [8]:
# the pairs of individuals (Parents and CHildren) whose labels we expect to be informative of each other
pch_pairs = [('mz','mzd'),('mz','mzs'),('mb','mbd'),('mb','mbs'),('m','z'),('m','b'),('f','z'),('f','b'),('fz','fzd'),('fz','fzs'),('fb','fbd'),('fb','fbs')]

In [9]:
# pair up individuals for joint probability

def get_pairs(ks: dict) -> list:
    pairs_of_terms = []
    pairs_of_relatives = []

    for pair in pch_pairs:
        if pair[0] in ks and pair[1] in ks:
            pairs_of_terms.append((ks[pair[0]],ks[pair[1]]))
            pairs_of_relatives.append((pair[0],pair[1]))
                            
    return pairs_of_terms,pairs_of_relatives

In [10]:
# split the pairs to work out which generation the term belongs to

def split_pairs(pairs_of_terms: list) -> list:
    """This function takes the pairs_of_terms variable created by get_pairs and splits the terms up so we know what
    generation they belong to. We know the 0th element in the pair will always be the parent and the 1st the child."""
    
    gn = []
    gn1 = []
    for pair in pairs_of_terms: # for each pair of terms
        gn.append(pair[1]) # append the 1st element in the pair to the Generation N list
        gn1.append(pair[0]) # append the 0th element in the pair to the Generation N+1 list
    
    return gn,gn1

In [30]:
def probability(term: str, generation: list) -> float:
    """Calculate the probability of a term given how common it is in its generation."""
    return generation.count(term)/len(generation)

def joint_probability(term1: str, term2: str, pairs: list) -> float:
    """Calculate the joint probability of two terms - how commonly they are paired together."""
    pair = (term1,term2)
    return pairs.count(pair)/len(pairs)

def joint_probability_distribution(ks):
    
    terms,relatives = get_pairs(ks)
        
    probs = {k:v for (k,v) in zip(terms, [0 for i in range(len(terms))])}  
    

    for i in range(len(terms)):
            probs[terms[i]] += 1/len(relatives)

    return probs

In [82]:
# calculate the entropy of terms in generation X given the terms in generation Y
def conditional_entropy(ks,x,y):
    """Calculate the summed entropy of all pairs of kin terms in the system. 
    X and Y indicate which variable conditions which."""
    entropy = 0
    
    pairs_of_terms,relatives = get_pairs(ks)
    g0,g1 = split_pairs(pairs_of_terms)
    
    probabilities = joint_probability_distribution(ks)
    
    for parent,child in set(pairs_of_terms): # for each pair of terms; x = parent, y = child
        p_xy = probabilities[parent,child] # calculate their joint probability
        p_ch = probability(child,g0) # calculate the probability of the child term in its generation
        p_par = probability(parent,g1) # calculate the probability of the parent term in its generation
        
        if x == 0: # if we want ce of child term given parent term (H(child|parent)):
            entropy += p_xy * math.log2(round(p_xy/p_par,3))
#             print('conditional prob of ' + child + ' given ' + parent)
#             print(round(p_xy/p_par,3))
#             print('entropy = ', -p_xy * math.log2(round(p_xy/p_par,3)))
        if x == 1: # if we want ce of parent term given child term (H(parent|child)):
            entropy += p_xy * math.log2(round(p_xy/p_ch,3))
#             print('conditional prob of ' + parent + ' given ' + child)
#             print(round(p_xy/p_ch,3))
#             print('entropy = ', -p_xy * math.log2(round(p_xy/p_ch,3)))
            
    return round(-entropy,5)

In [83]:
def predictive_structure(ks):
    """Calculate the sum of H(X|Y) and H(Y|X)."""
    
    hxy = conditional_entropy(ks,0,1)
    hyx = conditional_entropy(ks,1,0)
    
    return hxy + hyx

In [84]:
ks = {
    'm': 'mama',
    'f': 'ise',
    'fb': 'isento',
    'fz': 'isenkati',
    'mb': 'ongundwe',
    'mz': 'nyinento',
    'fbs':'munyanya',
    'fbd':'munyanya',
    'fzs':'mwhiwha',
    'fzd':'mwhiwha',
    'mbs':'nyinarumi',
    'mbd': 'nyinento',
    'mzs': 'omwana',
    'mzd': 'omwana',
    'z': 'mukuru',
    'b': 'mukuru'
}

In [85]:
predictive_structure(ks)

0.5

In [86]:
test_df = get_ppt_df(random_ids[3])

[test_df['generaliser1'].iloc[0],test_df['generaliser2'].iloc[0]]

# system_df['generaliser1'].iloc[0],system_df['generaliser2'].iloc[0]

['mzs', 'fbd']

Now, let's save a dataframe that codes each participants' responses for whether or not they generalised predictively.

In [87]:
def accuracy(filename):
    
    data = []
    
    for ppt_id in random_ids:
        df = get_ppt_df(ppt_id)
        
        try:
            df.empty == False
        except:
            continue
            
        for system in [0,1,2,3,4,5,6,7]:
            
            system_df = df.drop(df[df.system != system].index)
            
            if system_df.empty:
                continue
                
            else:
                
#                 print(system_df)

                if True in set(system_df['catch_failed']): # if they failed a catch trial, don't look at their data
                    pass

                else:
                    generalisers = [system_df['generaliser1'].iloc[0],system_df['generaliser2'].iloc[0]]

                    gen_terms = [system_df['generaliser1_term'].iloc[0],system_df['generaliser2_term'].iloc[0]]

                    ks = {}

                    ks[generalisers[0]] = gen_terms[0] # set the first and second generalisers
                    ks[generalisers[1]] = gen_terms[1]

                    choices = [gen_terms[0],gen_terms[1]]

                    ks = make_ks(language_types[system],ks) # fill the kinship system as the participants saw it to begin with

                    for row in range(len(system_df)):

                        chosen_term = system_df['selection'].values[row]

                        selectee = system_df['selectee'].values[row]

                        ks[selectee] = chosen_term  # for each choice they made, add it to the ks incrementally

                        if ks[selectee] not in choices:
                            choices.append(ks[selectee])

                        hypothetical = ks
                        possible_pss = []

                        for choice in choices:
                            hypothetical[selectee] = choice
                            hyp_ps = predictive_structure(hypothetical)
                            possible_pss.append(hyp_ps)

                        new_ps = predictive_structure(ks)

                        if new_ps == min(possible_pss):
                            accuracy = 1
                        else:
                            accuracy = 0


                        results = {}

                        results['random_id'] = system_df['random_id'].values[0]
                        results['condition'] = system_df['condition'].values[0]
                        results['block'] = system_df['block'].values[0]
                        results['system'] = system_df['system'].values[0]
                        results['selectee'] = selectee
                        results['chosen_term'] = chosen_term
                        results['gen1'] = generalisers[0]
                        results['gen1_term'] = gen_terms[0]
                        results['gen2'] = generalisers[1]
                        results['gen2_term'] = gen_terms[1]
                        results['correct'] = accuracy

                        data.append(results)

    pd.DataFrame(data).to_csv('data/' + filename + '.csv',index=False) # save the dataframe

    return pd.DataFrame(data)

In [88]:
accuracy('accuracy_equal')

       random_id condition  block  system trial_type selectee selection  \
2585  1mxy428t9t      mfgb      5       0      catch        z    hyonăo   
2586  1mxy428t9t      mfgb      5       0   critical      mbd    goozăi   
2587  1mxy428t9t      mfgb      5       0   critical      mzs    goozăi   
2588  1mxy428t9t      mfgb      5       0   critical      fzs    goozăi   
2589  1mxy428t9t      mfgb      5       0   critical      mzd    goozăi   
2590  1mxy428t9t      mfgb      5       0   critical      fbd    goozăi   
2591  1mxy428t9t      mfgb      5       0   critical        b    goozăi   
2592  1mxy428t9t      mfgb      5       0   critical      fzd    goozăi   
2593  1mxy428t9t      mfgb      5       0   critical      fbs    goozăi   

     expected  correct  undo  ...  generaliser2_term generaliser3  \
2585   hyonăo        1     0  ...             hyonăo          NaN   
2586   goozăi        1     0  ...             hyonăo          NaN   
2587   goozăi        1     0  ...         

      random_id condition  block  system trial_type selectee selection  \
258  zmpvzne4pl      fmbg      6       6   critical      mzd       gêă   
259  zmpvzne4pl      fmbg      6       6   critical      mzs    nyumuo   
260  zmpvzne4pl      fmbg      6       6   critical      fzs    nyumuo   
261  zmpvzne4pl      fmbg      6       6   critical      mbs    nyumuo   
262  zmpvzne4pl      fmbg      6       6      catch      mbd    niuhiê   
263  zmpvzne4pl      fmbg      6       6   critical      fbs    nyumuo   
264  zmpvzne4pl      fmbg      6       6   critical      fzd       gêă   

    expected  correct  undo  ...  generaliser2_term generaliser3  \
258      gêă        1     0  ...                gêă          NaN   
259      gêă        0     0  ...                gêă          mzs   
260      gêă        0     0  ...                gêă          mzs   
261   niuhiê        0     0  ...                gêă          mzs   
262   niuhiê        1     0  ...                gêă          mzs   

       random_id condition  block  system trial_type selectee selection  \
1538  n82s617h3t      mfbg      5       5      catch      fzd  orinoçim   
1539  n82s617h3t      mfbg      5       5   critical      fbs      iqur   
1540  n82s617h3t      mfbg      5       5   critical      fbd  orinoçim   
1541  n82s617h3t      mfbg      5       5   critical      mbd      iqur   
1542  n82s617h3t      mfbg      5       5   critical      mbs  içiqusem   
1543  n82s617h3t      mfbg      5       5   critical      mzs  içiqusem   
1544  n82s617h3t      mfbg      5       5   critical      fzs    adeçim   

      expected  correct  undo  ...  generaliser2_term generaliser3  \
1538  orinoçim        1     0  ...               iqur          NaN   
1539      iqur        1     0  ...               iqur          NaN   
1540      iqur        0     0  ...               iqur          NaN   
1541      iqur        1     0  ...               iqur          NaN   
1542      iqur        0     0  ...               

[7 rows x 24 columns]
       random_id condition  block  system trial_type selectee selection  \
2837  wh08dqfy5x      mfbg      1       6   critical      mbs    hïlaki   
2838  wh08dqfy5x      mfbg      1       6   critical      fbs    zuvewu   
2839  wh08dqfy5x      mfbg      1       6   critical      fzs    zuvewu   
2840  wh08dqfy5x      mfbg      1       6   critical      mzs    hïlaki   
2841  wh08dqfy5x      mfbg      1       6   critical      mzd    hïlaki   
2842  wh08dqfy5x      mfbg      1       6      catch      fzd    zuvewu   
2843  wh08dqfy5x      mfbg      1       6   critical      fbd    zuvewu   

     expected  correct  undo  ...  generaliser2_term generaliser3  \
2837   hïlaki        1     0  ...             zuvewu          NaN   
2838   zuvewu        1     0  ...             zuvewu          NaN   
2839   zuvewu        1     0  ...             zuvewu          NaN   
2840   zuvewu        0     0  ...             zuvewu          NaN   
2841   zuvewu        0     0  ..

       random_id condition  block  system trial_type selectee selection  \
1210  96dcxvnjng      mfgb      7       4   critical      fbs    adeçim   
1211  96dcxvnjng      mfgb      7       4   critical      fzs    adeçim   
1212  96dcxvnjng      mfgb      7       4   critical      fzd    adeçim   
1213  96dcxvnjng      mfgb      7       4   critical      mzs    adeçim   
1214  96dcxvnjng      mfgb      7       4   critical      mbd    adeçim   
1215  96dcxvnjng      mfgb      7       4      catch      fbd    adeçim   
1216  96dcxvnjng      mfgb      7       4   critical      mzd    adeçim   

      expected  correct  undo  ...  generaliser2_term generaliser3  \
1210    adeçim        1     0  ...           orinoçim          NaN   
1211  orinoçim        0     0  ...           orinoçim          NaN   
1212  orinoçim        0     0  ...           orinoçim          NaN   
1213  orinoçim        0     0  ...           orinoçim          NaN   
1214  orinoçim        0     0  ...           orin

...

       random_id condition  block  system trial_type selectee selection  \
1958  eeyykmwnln      fmbg      2       6   critical      mbd    goviku   
1959  eeyykmwnln      fmbg      2       6   critical      fbd    lawolu   
1960  eeyykmwnln      fmbg      2       6   critical      mzd    lawolu   
1961  eeyykmwnln      fmbg      2       6   critical      fzs    lawolu   
1962  eeyykmwnln      fmbg      2       6   critical      mzs    lawolu   
1963  eeyykmwnln      fmbg      2       6   critical      fzd    lawolu   
1964  eeyykmwnln      fmbg      2       6      catch      mbs    goviku   

     expected  correct  undo  ...  generaliser2_term generaliser3  \
1958   goviku        1     0  ...             lawolu          NaN   
1959   lawolu        1     0  ...             lawolu          NaN   
1960   lawolu        1     0  ...             lawolu          NaN   
1961   lawolu        1     0  ...             lawolu          NaN   
1962   lawolu        1     0  ...             lawolu  

      random_id condition  block  system trial_type selectee selection  \
768  4k1ad46eg9      fmbg      5       3   critical      fzd    kawuka   
769  4k1ad46eg9      fmbg      5       3   critical      mbs      wupu   
770  4k1ad46eg9      fmbg      5       3   critical      fbs      wupu   
771  4k1ad46eg9      fmbg      5       3      catch      mbd    kawuka   
772  4k1ad46eg9      fmbg      5       3   critical      fbd    kawuka   
773  4k1ad46eg9      fmbg      5       3   critical      mzd    kawuka   
774  4k1ad46eg9      fmbg      5       3   critical      fzs      wupu   

    expected  correct  undo  ...  generaliser2_term generaliser3  \
768   kawuka        1     0  ...               wupu          NaN   
769   kawuka        0     0  ...               wupu          NaN   
770     wupu        1     0  ...               wupu          NaN   
771   kawuka        1     0  ...               wupu          NaN   
772     wupu        0     0  ...               wupu          NaN   

[7 rows x 24 columns]
       random_id condition  block  system trial_type selectee selection  \
2777  srcyko4mfl      fmbg      2       2   critical      mbd     byure   
2778  srcyko4mfl      fmbg      2       2      catch      mbs    cžunge   
2779  srcyko4mfl      fmbg      2       2   critical      mzd     byure   
2780  srcyko4mfl      fmbg      2       2   critical      fzd     vruži   
2781  srcyko4mfl      fmbg      2       2   critical      fbs     vruži   
2782  srcyko4mfl      fmbg      2       2   critical      fbd     vruži   
2783  srcyko4mfl      fmbg      2       2   critical      mzs    cžunge   

     expected  correct  undo  ...  generaliser2_term generaliser3  \
2777   cžunge        0     0  ...              vruži          mbd   
2778   cžunge        1     0  ...              vruži          mbd   
2779    vruži        0     0  ...              vruži          mbd   
2780    vruži        1     0  ...              vruži          mbd   
2781   cžunge        0     0  ..

       random_id condition  block  system trial_type selectee  selection  \
1316  by11zvzmpo      mfgb      4       5      catch      fzs  powisunge   
1317  by11zvzmpo      mfgb      4       5   critical      fbd       luge   
1318  by11zvzmpo      mfgb      4       5   critical      fbs  powisunge   
1319  by11zvzmpo      mfgb      4       5   critical      mbs  powisunge   
1320  by11zvzmpo      mfgb      4       5   critical      mzs  powisunge   
1321  by11zvzmpo      mfgb      4       5   critical      fzd       luge   
1322  by11zvzmpo      mfgb      4       5   critical      mzd       luge   

       expected  correct  undo  ...  generaliser2_term generaliser3  \
1316  powisunge        1     0  ...               luge          NaN   
1317       luge        1     0  ...               luge          NaN   
1318       luge        0     0  ...               luge          NaN   
1319       luge        0     0  ...               luge          NaN   
1320       luge        0     0  ... 

,random_id,condition,block,system,selectee,chosen_term,gen1,gen1_term,gen2,gen2_term,correct
0,1mxy428t9t,mfgb,5,0,z,hyonăo,mbs,goozăi,z,hyonăo,1
1,1mxy428t9t,mfgb,5,0,mbd,goozăi,mbs,goozăi,z,hyonăo,0
2,1mxy428t9t,mfgb,5,0,mzs,goozăi,mbs,goozăi,z,hyonăo,0
3,1mxy428t9t,mfgb,5,0,fzs,goozăi,mbs,goozăi,z,hyonăo,0
4,1mxy428t9t,mfgb,5,0,mzd,goozăi,mbs,goozăi,z,hyonăo,1
...,...,...,...,...,...,...,...,...,...,...,...
2203,wa2j69ws6s,fmbg,1,7,mzs,luge,mzd,lepego,fbd,luge,0
2204,wa2j69ws6s,fmbg,1,7,fzs,luge,mzd,lepego,fbd,luge,1
2205,wa2j69ws6s,fmbg,1,7,fzd,lepego,mzd,lepego,fbd,luge,1
2206,wa2j69ws6s,fmbg,1,7,mbs,luge,mzd,lepego,fbd,luge,1


# By-participant baselines and z-scores

Creating a baseline distribution for participant responses - are participants generalising more predictively than we would expect by chance?

Shuffle the participants' responses - break all predictive bonds, but maintain number of terms used and entropy of each term.

In [89]:
def shuffle_ppt_responses(df):
        
    ks = {}
    
    choices = []
    catcher = ""
    
    for i in range(len(df)):
        relative = df['selectee'].values[i] # get the referent for each trial
        
        if df['trial_type'].values[i] == 'catch': # ignore if it's a catch trial
            continue

        else:
            term = df['selection'].values[i] # get the term the participant chose for this referent
            ks[relative] = term # add it to the dict
            
            
    terms = list(ks.values()) # get all the terms the participant used
    
    random.shuffle(terms) # and shuffle them
    
    # then reassign them to referents in this random order
    index = 0
    for i in ks:
        ks[i] = terms[index]
        index += 1
    
    # assign the generalisers to the dict - these don't get shuffled
    generaliser1 = df['generaliser1'].values[0]
    ks[generaliser1] = df['generaliser1_term'].values[0]
    generaliser2 = df['generaliser2'].values[0]
    ks[generaliser2] = df['generaliser2_term'].values[0]

    return ks

Shuffle each participant output a number of times and compare to the true value - save the z-score to a CSV file.

In [90]:
def by_participant_baselines(filename,filename2,times):
    
    data = []
    z_data = []
    
    for ppt_id in random_ids:
        df = get_ppt_df(ppt_id)
        
        try:
            df.empty == False
        except:
            continue
            
        for system in [0,1,2,3,4,5,6,7]:

            system_df = df.drop(df[df.system != system].index)
            
            if system_df.empty:
                continue
                
            else:

                if True in set(system_df['catch_failed']):
                    pass

                else:

                    block = system_df['block'].values[0]
                    indx = system_df[system_df.trial_type == 'catch'].index[0]
                    system_df = system_df.drop(indx)

                    ppt_responses = get_ppt_ks(system_df)
                    true_ks = make_ks(language_types[system],ppt_responses)

                    true_value = predictive_structure(true_ks)

                    sim_values = []

                    for j in range(times):

                        results = {}

                        shuffled_responses = shuffle_ppt_responses(system_df)

                        sim = make_ks(language_types[system],shuffled_responses)

                        ps = predictive_structure(sim)

                        sim_values.append(ps)

                        results['random_id'] = ppt_id
                        results['block'] = block
                        results['system'] = system
                        results['simulated_value'] = ps
                        results['true_value'] = true_value
                        results['data_type'] = 'simulation'

                        for relative in sim:
                            results[relative] = sim[relative]

                        data.append(results)

                    z_results = {}

                    mean = np.mean(sim_values)
                    sd = np.std(sim_values)

                    z_results['random_id'] = ppt_id
                    z_results['block'] = block
                    z_results['system'] = system
                    z_results['true_value'] = true_value
                    z_results['simulated_mean'] = mean
                    z_results['simulated_sd'] = sd
                    z_results['z'] = (true_value - mean) / sd

                    z_data.append(z_results)

        
    pd.DataFrame(data).to_csv('data/' + filename + '.csv',index=False) # save the dataframe
    pd.DataFrame(z_data).to_csv('data/' + filename2 + '.csv',index=False) # save the dataframe


            
    return pd.DataFrame(data)

In [91]:
by_participant_baselines('by_participant_baselines_equal','by_participant_z_equal',1000)

C:\Users\s1604058\AppData\Local\Temp\ipykernel_10212\3986574290.py:74: RuntimeWarning: invalid value encountered in scalar divide
  z_results['z'] = (true_value - mean) / sd
C:\Users\s1604058\AppData\Local\Temp\ipykernel_10212\3986574290.py:74: RuntimeWarning: invalid value encountered in scalar divide
  z_results['z'] = (true_value - mean) / sd
C:\Users\s1604058\AppData\Local\Temp\ipykernel_10212\3986574290.py:74: RuntimeWarning: invalid value encountered in scalar divide
  z_results['z'] = (true_value - mean) / sd
C:\Users\s1604058\AppData\Local\Temp\ipykernel_10212\3986574290.py:74: RuntimeWarning: invalid value encountered in scalar divide
  z_results['z'] = (true_value - mean) / sd
C:\Users\s1604058\AppData\Local\Temp\ipykernel_10212\3986574290.py:74: RuntimeWarning: invalid value encountered in scalar divide
  z_results['z'] = (true_value - mean) / sd


,random_id,block,system,simulated_value,true_value,data_type,mbd,mzs,fzs,mzd,...,fzd,fbs,mbs,z,m,f,mz,mb,fz,fb
0,1mxy428t9t,5,0,1.26827,1.26827,simulation,goozăi,goozăi,goozăi,goozăi,...,goozăi,goozăi,goozăi,hyonăo,0,1,2,2,2,2
1,1mxy428t9t,5,0,1.26827,1.26827,simulation,goozăi,goozăi,goozăi,goozăi,...,goozăi,goozăi,goozăi,hyonăo,0,1,2,2,2,2
2,1mxy428t9t,5,0,1.26827,1.26827,simulation,goozăi,goozăi,goozăi,goozăi,...,goozăi,goozăi,goozăi,hyonăo,0,1,2,2,2,2
3,1mxy428t9t,5,0,1.26827,1.26827,simulation,goozăi,goozăi,goozăi,goozăi,...,goozăi,goozăi,goozăi,hyonăo,0,1,2,2,2,2
4,1mxy428t9t,5,0,1.26827,1.26827,simulation,goozăi,goozăi,goozăi,goozăi,...,goozăi,goozăi,goozăi,hyonăo,0,1,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284995,wa2j69ws6s,1,7,1.48952,1.48952,simulation,luge,luge,luge,lepego,...,lepego,luge,lepego,4,0,1,3,2,2,2
284996,wa2j69ws6s,1,7,1.48952,1.48952,simulation,lepego,luge,lepego,lepego,...,luge,luge,luge,4,0,1,3,2,2,2
284997,wa2j69ws6s,1,7,1.48952,1.48952,simulation,lepego,luge,lepego,lepego,...,luge,luge,luge,4,0,1,3,2,2,2
284998,wa2j69ws6s,1,7,0.88792,1.48952,simulation,lepego,lepego,luge,lepego,...,luge,luge,luge,4,0,1,3,2,2,2


# Semantic distance analysis

To generate the heatmap plots in Figure 14, we need to calculate the proportion with which each pair of referents was put into the same category (i.e. participants assigned them the same term).

In [34]:
selectees = ['mbs','mbd','mzs','mzd','fbs','fbd','fzs','fzd','z','b']

In [ ]:
def calculate_distance():
    

    for system in [0,1,2,3,4,5,6,7]:
    
        languages = []
        
        for ppt_id in random_ids:
            df = get_ppt_df(ppt_id)
                    
            # first, get all participant kinship systems
            try:
                if df.empty == False:

                    system_df = df.drop(df[df.system != system].index)

                    if True in set(system_df['catch_failed']):
                        pass
                    else:
                        ppt_responses = get_ppt_ks(system_df)
                        languages.append(ppt_responses)
            except:
                continue
    
            data = []

            # then, for each pair of relatives, score 1 if they were given the same label

            for relative1 in selectees:
                for relative2 in selectees:
                    scores = []
                    for i in range(len(languages)):
                        if relative1 in languages[i] and relative2 in languages[i]:
                            if languages[i][relative1] == languages[i][relative2]:
                                score = 1
                            else:
                                score = 0
                            scores.append(score)
                        else: 
                            continue
                    try:        
                        dist = sum(scores)/len(scores)
                    except:
                        dist = 0

        #             print(sum(scores),len(scores))

                    results = {}
                    results['x'] = relative1
                    results['y'] = relative2
                    results['z'] = dist
                    results['system'] = system
                    results['system_type'] = systems[system]
                    results['system_group'] = system_groups[system]
                    results['gender_distinction'] = gender_distinction

                    data.append(results)
    
    pd.DataFrame(data).to_csv('data/' + str(system) + '_distance.csv',index=False) # save the dataframe
    
    return pd.DataFrame(data)

In [29]:
# calculate_distance(0)

The following builds a dataframe that instantiates what our data would look like in an ideal world, i.e. if participants categorised kin predictively 100% of the time (for the bottom row of Figure 14):

In [32]:
perfect_languages = [[['z','b'],['mzd','mzs','mbd','mbs','fzd','fzs','fbd','fbs']], # Type I
                [['mzd','mzs','mbd','mbs'],['fzd','fzs','fbd','fbs']], # Type II side
                [['mzd','mzs','fzd','fzs'],['mbd','mbs','fbd','fbs']], # Type II gen
                [['mzd','mzs','fbd','fbs'],['mbd','mbs','fzd','fzs']], # Type II opp
                [['mzd','mzs','mbd','mbs','fzd','fzs'],['fbd','fbs']], # Type III fb
                [['mzd','mzs','mbd','mbs','fbd','fbs'],['fzd','fzs']], # Type III fz
                [['mzd','mzs','fzd','fzs','fbd','fbs'],['mbd','mbs']], # Type III mb
                [['mbd','mbs','fzd','fzs','fbd','fbs'],['mzd','mzs']]] # Type III mz

In [35]:
def perfect_distance(perfect_languages,filename):
    
    df = []
    for language in perfect_languages:
        
        for relative1 in selectees:
            for relative2 in selectees:
                data = {}
                if relative1 in language[0] and relative2 in language[0]:
                    data['x'] = relative1
                    data['y'] = relative2
                    data['z'] = 1
                    
                elif relative1 in language[1] and relative2 in language[1]:
                    data['x'] = relative1
                    data['y'] = relative2
                    data['z'] = 1
                else:
                    data['x'] = relative1
                    data['y'] = relative2
                    data['z'] = 0.1
                    
                system = perfect_languages.index(language)
                data['system'] = system
                data['system_type'] = systems[system]
                data['system_group'] = system_groups[system]
                
                df.append(data)
                
#     pd.DataFrame(df).to_csv('data/' + filename + '.csv',index=False) # save the dataframe
    
    return pd.DataFrame(df)

# Gender analysis

Our participants often encode gender. For some exploratory analysis, let's calculate the proportion of gender distinctions made by our participants.

In [37]:
boys = ['mbs','mzs','fbs','fzs','b']
girls = ['mbd','mzd','fbd','fzd','z']

gender_pairs = [['mbs','mbd'],['mzs','mzd'],['fbs','fbd'],['fzs','fzd'],['b','z']]

In [39]:
def gender_distinctions(filename):
    data = []
    
    for ppt_id in random_ids:
        
        df = get_ppt_df(ppt_id)
        
        try:
            df.empty == False
        except:
            continue
        
        for system in [0,1,2,3,4,5,6,7]:

            system_df = df.drop(df[df.system != system].index)

            if True in set(system_df['catch_failed']):
                pass

            else:

                ppt_responses = get_ppt_ks(system_df)
                
                for pair in gender_pairs:
                    results = {}
                    
                    boy = pair[0]
                    girl = pair[1]
                    if boy in ppt_responses and girl in ppt_responses:
                        if ppt_responses[boy] == ppt_responses[girl]:
                            gen = 0
                        else:
                            gen = 1
                
#                 for relative1 in ppt_responses:
#                     for relative2 in ppt_responses:
#                             if relative1 in boys and relative2 in girls:
#                                 if ppt_responses[relative1] == ppt_responses[relative2]:
#                                     gen = 0
#                                 else:
#                                     gen = 1
                        results['random_id'] = system_df['random_id'].values[0]
                        results['condition'] = system_df['condition'].values[0]
                        results['block'] = system_df['block'].values[0]
                        results['system'] = system_df['system'].values[0]
                        results['relative1'] = boy
                        results['relative2'] = girl
                        results['relative1_term'] = ppt_responses[boy]
                        results['relative2_term'] = ppt_responses[girl]
                        results['distinction'] = gen
                            

                        data.append(results)
                    
    pd.DataFrame(data).to_csv('data/' + filename + '.csv',index=False) # save the dataframe
                    
    return pd.DataFrame(data)
    

In [ ]:
gender_distinctions('gender_distinctions')